<a href="https://colab.research.google.com/github/jottanovaes/colab-notebooks/blob/trabalho-final-programacao-linear/trabalho_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema 6 - Escala de funcionários
Uma empresa possui uma equipe que trabalha sete dias por semana em turnos de trabalho de
mesmo tempo.

A equipe possui *n* funcionários. Cada funcionário deve ter 5 folgas durante o
mês e para garantir que o trabalho seja realizado de forma competente todos os dias, a equipe
não pode operar com menos que *l* funcionários.

Por razões contratuais nenhum funcionário pode
trabalhar por 6 dias seguidos sem folgas e ao menos uma das folgas deve ocorrer em um domingo
ou em um feriado. Os funcionários são consultados e indicam datas preferenciais para folgas.
Construa um modelo de otimização que determine a escala dos funcionários para o mês de junho
de 2026 que:

1.   Maximize o número total de folgas nos dias requisitados pelos funcionários.
2.   Distribua as folgas nos dias requisitados de maneira mais equilibrada possível entre os funcionários.

# 1. Maximize o número total de folgas nos dias requisitados pelos funcionários.

## modelo algebrico
- *n* funcionarios
- *m* dia do mês

## variaveis de decisao
$$
x_{ij} =
\begin{cases}
1, & \text{se o funcionário folga no dia} \\
0, & \text{caso contrário}
\end{cases}
$$

## parametros
$$ l = \text{minimo de funcionarios em cada turno} $$
$$ D = \text{domingos e feriados} $$
$$ P_{ij} = \text{preferencia do funcionario } i \text{ para folgar no dia } j $$

## funcao objetivo
$$
 max  \sum_{i=1}^{n}\sum_{j=1}^{m} P_{ij} \cdot x_{ij}
$$

## retricoes
$$
\sum_{j=1}^{m} x_{ij} = 5, \quad \forall i \in \{1, \ldots, n\}
$$

$$
\sum_{i=1}^{n} (1 - x_{ij}) \ge l, \quad \forall j \in \{1, \ldots, m\}
$$

$$
\sum_{j \in D} x_{ij} \ge 1, \quad \forall i \in \{1, \ldots, n\}
$$

$$
\sum_{k=j}^{j+5} x_{ik} \ge 1, \quad \forall i \in \{1,\ldots,n\},\ \forall j \in \{1, \ldots, m-5\}
$$

$$
x_{ij} \in \{0, 1\}, \quad \forall i, \forall j
$$

# 2. Distribua as folgas nos dias requisitados de maneira mais equilibrada possível entre os funcionários.

Esse modelo mantém as mesmas restrições do modelo A e adiciona restrições para equilibrar as folgas

Será criado duas variáveis *z_max* e *z_min*

O objetivo será alterado

$$
min (z_{max} - z_{min} )
$$

In [55]:
!pip install gurobipy

import pandas as pd
import requests
import gurobipy as gp
import time
import numpy as np
from google.colab import files

In [56]:
# calendario junho/2026
D = {3, 6, 13, 20, 27} # indice dos dias que sao domingos o feriados
m = 30

# dados de entrada
instancias = [
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_20_15.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_25_19.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_30_23.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_35_27.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_40_31.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_45_35.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_50_39.txt",
    "https://raw.githubusercontent.com/jottanovaes/colab-notebooks/refs/heads/trabalho-final-programacao-linear/inst_60_47.txt",
]

In [57]:
def ler_instancia(url):
    texto = requests.get(url).text
    linhas = [l.strip() for l in texto.strip().splitlines() if l.strip()]

    n = int(linhas[0]) # numero de funcionarios
    l = int(linhas[1]) # numero minimo de funcionarios em cada turno

    P = [] # preferencias
    for i in range(2, 2 + n):
        dias = [int(d) - 1 for d in linhas[i].split()]
        P.append(dias)

    P_matrix = [
        [1 if j in P[i] else 0 for j in range(m)]
        for i in range(n)
    ] # cria matriz binária n x m, 1 se o funcionario i quer folgar no dia j


    return n, l, P, P_matrix

In [58]:
def otimizar_a(n, l, m, P, P_matrix, D):
  modelo = gp.Model()

  x = modelo.addVars(
      n,
      m,
      vtype = gp.GRB.BINARY,
      name = "x"
  )

  modelo.setObjective(
      gp.quicksum(P_matrix[i][j] * x[i, j] for i in range(n) for j in range(m)),
      gp.GRB.MAXIMIZE
  )

  # r1: cada funcionario tem exatamente 5 folgas no mes
  r1 = modelo.addConstrs(
      gp.quicksum(x[i, j] for j in range(m)) == 5
      for i in range(n)
  )

  # r2: minimo de l funcionarios trabalhando por dia
  r2 = modelo.addConstrs(
      gp.quicksum(1 - x[i, j] for i in range(n)) >= l
      for j in range(m)
  )

  # r3: pelo menos uma folga em domingo ou feriado
  r3 = modelo.addConstrs(
      gp.quicksum(x[i, j] for j in D) >= 1
      for i in range(n)
  )

  # r4: nenhum funcionario trabalha 6 dias seguidos sem folga
  # para uma janela de 7 dias, pelo menos 1 folga
  r4 = modelo.addConstrs(
      gp.quicksum(x[i, k] for k in range(j, j + 7)) >= 1
      for i in range(n)
      for j in range(m - 6)
  )

  t0 = time.time()

  modelo.setParam("OutputFlag", 0)
  modelo.optimize()
  tempo = round(time.time() - t0, 3)

  return modelo, x, tempo

In [59]:
def otimizar_b(n, l, m, P, P_matrix, D):
    modelo = gp.Model()
    modelo.setParam("OutputFlag", 0)

    x     = modelo.addVars(n, m, vtype=gp.GRB.BINARY, name="x")
    z_max = modelo.addVar(lb=0, ub=4, vtype=gp.GRB.INTEGER, name="z_max")
    z_min = modelo.addVar(lb=0, ub=4, vtype=gp.GRB.INTEGER, name="z_min")

    # objetivo: minimizar desequilíbrio entre funcionários
    modelo.setObjective(z_max - z_min, gp.GRB.MINIMIZE)

    # R1–R4 idênticas ao modelo A
    modelo.addConstrs(
        gp.quicksum(x[i, j] for j in range(m)) == 5
        for i in range(n)
    )
    modelo.addConstrs(
        gp.quicksum(x[i, j] for i in range(n)) <= n - l
        for j in range(m)
    )
    modelo.addConstrs(
        gp.quicksum(x[i, k] for k in range(j, j + 7)) >= 1
        for i in range(n)
        for j in range(m - 6)
    )
    modelo.addConstrs(
        gp.quicksum(x[i, j] for j in D) >= 1
        for i in range(n)
    )

    # R5: z_max e z_min capturam os extremos de preferências atendidas
    for i in range(n):
        soma_pref_i = gp.quicksum(P_matrix[i][j] * x[i, j] for j in range(m))
        modelo.addConstr(z_max >= soma_pref_i)
        modelo.addConstr(z_min <= soma_pref_i)

    t0 = time.time()
    modelo.optimize()
    tempo = round(time.time() - t0, 3)

    return modelo, x, z_max, z_min, tempo

In [60]:
rows_func = []
rows_inst = []

for url in instancias:
    nome = url.split("/")[-1].replace(".txt", "")
    print(f"\n── {nome} ──")

    n, l, P, P_matrix = ler_instancia(url)

    for modelo_nome, resultado in [
        ("A", lambda: otimizar_a(n, l, m, P, P_matrix, D)),
        ("B", lambda: otimizar_b(n, l, m, P, P_matrix, D)),
    ]:
        if modelo_nome == "A":
            modelo, x, tempo = resultado()
            z_max_val = z_min_val = desequilibrio = None
        else:
            modelo, x, z_max_var, z_min_var, tempo = resultado()
            z_max_val     = round(z_max_var.X) if modelo.status == gp.GRB.OPTIMAL else None
            z_min_val     = round(z_min_var.X) if modelo.status == gp.GRB.OPTIMAL else None
            desequilibrio = (z_max_val - z_min_val) if z_max_val is not None else None

        if modelo.status == gp.GRB.OPTIMAL:
            status = "otimo"
            obj_valor = int(modelo.objVal)
            pcts = []

            for i in range(n):
                folgas     = [j + 1 for j in range(m) if round(x[i, j].x) == 1]
                preferidas = [j + 1 for j in range(m) if round(x[i, j].x) == 1 and j in P[i]]
                total_pref = len(P[i])
                pct        = round(len(preferidas) / total_pref * 100, 1) if total_pref else 0.0
                tem_dom    = int(any(j in D for j in range(m) if round(x[i, j].x) == 1))
                pcts.append(pct)

                rows_func.append({
                    "instancia"             : nome,
                    "modelo"                : modelo_nome,
                    "n"                     : n,
                    "l"                     : l,
                    "funcionario"           : i + 1,
                    "folgas_atribuidas"     : ",".join(map(str, folgas)),
                    "preferencias_atendidas": ",".join(map(str, preferidas)),
                    "total_preferencias"    : total_pref,
                    "pct_atendida"          : pct,
                    "tem_folga_domingo"     : tem_dom,
                })

            rows_inst.append({
                "instancia"         : nome,
                "modelo"            : modelo_nome,
                "n"                 : n,
                "l"                 : l,
                "status"            : status,
                "obj_valor"         : obj_valor,
                "media_pct_atendida": round(np.mean(pcts), 2),
                "desvio_pct"        : round(np.std(pcts), 2),
                "min_pct"           : min(pcts),
                "max_pct"           : max(pcts),
                "z_max"             : z_max_val,
                "z_min"             : z_min_val,
                "desequilibrio"     : desequilibrio,
                "tempo_execucao_s"  : tempo,
            })

            print(f"  [{modelo_nome}] obj={obj_valor} | média%={rows_inst[-1]['media_pct_atendida']} | desvio={rows_inst[-1]['desvio_pct']}", end="")
            if modelo_nome == "B":
                print(f" | desequilíbrio={desequilibrio} (z_max={z_max_val}, z_min={z_min_val})", end="")
            print()

        else:
            print(f"  [{modelo_nome}] infactível")
            rows_inst.append({
                "instancia": nome, "modelo": modelo_nome,
                "n": n, "l": l, "status": "infactivel",
                "obj_valor": None, "media_pct_atendida": None,
                "desvio_pct": None, "min_pct": None, "max_pct": None,
                "z_max": None, "z_min": None, "desequilibrio": None,
                "tempo_execucao_s": tempo,
            })

# exporta
df_func = pd.DataFrame(rows_func)
df_inst = pd.DataFrame(rows_inst)

df_func.to_csv("resultados_funcionarios.csv", index=False, header=False)
df_inst.to_csv("resultados_instancias.csv",   index=False, header=False)

print("\n✓ resultados_funcionarios.csv")
files.download("resultados_funcionarios.csv")

print("✓ resultados_instancias.csv")
files.download("resultados_instancias.csv")

display(df_inst)


── inst_20_15 ──
  [A] obj=57 | média%=71.25 | desvio=16.35
  [B] obj=0 | média%=0.0 | desvio=0.0 | desequilíbrio=0 (z_max=0, z_min=0)

── inst_25_19 ──
  [A] obj=75 | média%=75.0 | desvio=15.81
  [B] obj=0 | média%=25.0 | desvio=0.0 | desequilíbrio=0 (z_max=1, z_min=1)

── inst_30_23 ──
  [A] obj=90 | média%=75.0 | desvio=15.81
  [B] obj=0 | média%=25.0 | desvio=0.0 | desequilíbrio=0 (z_max=1, z_min=1)

── inst_35_27 ──
  [A] obj=103 | média%=73.57 | desvio=14.57
  [B] obj=0 | média%=25.0 | desvio=0.0 | desequilíbrio=0 (z_max=1, z_min=1)

── inst_40_31 ──
  [A] obj=120 | média%=75.0 | desvio=16.77
  [B] obj=0 | média%=25.0 | desvio=0.0 | desequilíbrio=0 (z_max=1, z_min=1)

── inst_45_35 ──
  [A] obj=131 | média%=72.78 | desvio=13.77
  [B] obj=0 | média%=25.0 | desvio=0.0 | desequilíbrio=0 (z_max=1, z_min=1)

── inst_50_39 ──
  [A] obj=142 | média%=71.0 | desvio=15.3
  [B] obj=0 | média%=25.0 | desvio=0.0 | desequilíbrio=0 (z_max=1, z_min=1)

── inst_60_47 ──
  [A] obj=180 | média%=75

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ resultados_instancias.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,instancia,modelo,n,l,status,obj_valor,media_pct_atendida,desvio_pct,min_pct,max_pct,z_max,z_min,desequilibrio,tempo_execucao_s
0,inst_20_15,A,20,15,otimo,57,71.25,16.35,50.0,100.0,NaN,NaN,NaN,0.024
1,inst_20_15,B,20,15,otimo,0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.016
2,inst_25_19,A,25,19,otimo,75,75.00,15.81,50.0,100.0,NaN,NaN,NaN,0.040
3,inst_25_19,B,25,19,otimo,0,25.00,0.00,25.0,25.0,1.0,1.0,0.0,0.082
4,inst_30_23,A,30,23,otimo,90,75.00,15.81,50.0,100.0,NaN,NaN,NaN,0.045
5,inst_30_23,B,30,23,otimo,0,25.00,0.00,25.0,25.0,1.0,1.0,0.0,0.060
6,inst_35_27,A,35,27,otimo,103,73.57,14.57,50.0,100.0,NaN,NaN,NaN,0.058
7,inst_35_27,B,35,27,otimo,0,25.00,0.00,25.0,25.0,1.0,1.0,0.0,0.075
8,inst_40_31,A,40,31,otimo,120,75.00,16.77,50.0,100.0,NaN,NaN,NaN,0.033
9,inst_40_31,B,40,31,otimo,0,25.00,0.00,25.0,25.0,1.0,1.0,0.0,0.041
